[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atsyplenkov/ai_geoimage_coreg/blob/main/examples/default_coreg_colab_ru.ipynb)

# AI GeoImage Coreg - примерный запуск в Google Colab

**Рекомендуемая среда:** Google Colab с **GPU (T4)**.

Предполагается, что входные и выходные файлы в этом workflow хранятся на **Google Drive** (смонтированном в `/content/drive`).

## 1) Проверка GPU в среде выполнения

Если эта ячейка завершается ошибкой, переключите среду выполнения: **Runtime -> Change runtime type -> T4 GPU**, затем запустите заново.

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU is not enabled. In Colab: Runtime -> Change runtime type -> T4 GPU.')

## 2) Подключение Google Drive

Когда появится окно авторизации Drive, подтвердите доступ, чтобы подключить Google Colab к вашему Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Обязательные входные файлы

Загрузите два входных GeoTIFF-файла в `DRIVE_ROOT` (или обновите пути ниже):

- Историческое изображение: `historical.tif`
- Геопривязанное современное опорное изображение: `modern.tif`

Оба файла должны находиться на **Google Drive**.

## 3) Настройка путей входа и выхода на Drive

Определите, где на Google Drive находятся историческое изображение, современное опорное изображение и выходные результаты.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ai_geoimage_coreg')
HISTORICAL_IMAGE = DRIVE_ROOT / 'historical.tif'
REFERENCE_IMAGE = DRIVE_ROOT / 'modern.tif'
OUTPUT_PREFIX = DRIVE_ROOT / 'georef'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print('If your files are elsewhere in Drive, edit DRIVE_ROOT/HISTORICAL_IMAGE/REFERENCE_IMAGE above.')

## 4) Установка зависимостей в Colab

Установите системные пакеты GDAL и актуальный код репозитория из GitHub.

In [ ]:
!apt-get update
!apt-get install -y gdal-bin libgdal-dev
!pip install --upgrade pip
!pip install git+https://github.com/atsyplenkov/ai_geoimage_coreg.git

## 5) Проверка импортов

Проверьте, что `GDAL`, `Rasterio` и `run_pipeline` корректно импортируются перед запуском обработки.

In [ ]:
from osgeo import gdal
import rasterio
from ai_geoimage_coreg.core import run_pipeline

print('GDAL version:', gdal.VersionInfo())
print('Rasterio version:', rasterio.__version__)
print('Imports OK. If these imports fail after install, restart runtime and rerun cells.')

## 6) Проверка наличия входных файлов

Убедитесь, что оба входных TIFF-файла присутствуют перед запуском пайплайна.

In [ ]:
missing = [str(p) for p in [HISTORICAL_IMAGE, REFERENCE_IMAGE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing input files:\n'
        + '\n'.join(missing)
        + '\n\nUpload both TIFFs to DRIVE_ROOT or edit the path variables above.'
    )

print('Input files found.')

## 7) Запуск базовой ко-регистрации

Запустите пайплайн с параметрами по умолчанию и сохраните результаты на Google Drive.

In [ ]:
run_pipeline(
    path_hex=str(HISTORICAL_IMAGE),
    path_ref=str(REFERENCE_IMAGE),
    output_prefix=str(OUTPUT_PREFIX),
)

## 8) Проверка выходных файлов

Проверьте, что ожидаемые выходные файлы появились в вашей папке результатов на Google Drive.

In [ ]:
expected_outputs = [
    DRIVE_ROOT / 'georef_raw.csv',
    DRIVE_ROOT / 'georef_clean.csv',
    DRIVE_ROOT / 'georef_poly.tif',
    DRIVE_ROOT / 'georef_tps.tif',
]

for out in expected_outputs:
    print(f'{out.name}: {"OK" if out.exists() else "MISSING"}')